# 04 — Model Evaluation
## Loan Default Prediction System

**Purpose:** Independently evaluate the serialized `loan_pipeline.pkl` artifact on the held-out test set — confusion matrix, classification report, ROC-AUC, and feature importance — and persist the results to `ml/reports/evaluation_metrics.json`.

**Input:** `ml/models/loan_pipeline.pkl`, `ml/data/processed/test_set.csv`
**Output:** `ml/reports/evaluation_metrics.json`, `ml/reports/confusion_matrix.png`, `ml/reports/roc_curve.png`, `ml/reports/feature_importance.png`

This notebook reloads the saved pipeline from disk exactly as FastAPI's `model_loader.py` will, so the numbers here reflect the actual deployed artifact rather than an in-memory object from the training run.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = ML_ROOT / "src"
sys.path.append(str(SRC_PATH))

from evaluate_model import (
    load_artifact,
    load_test_set,
    compute_core_metrics,
    compute_confusion_matrix,
    compute_classification_report,
    extract_feature_importance,
    evaluate,
    MODEL_PATH,
    TEST_SET_PATH,
)

print(f"Model path: {MODEL_PATH}")
print(f"Test set path: {TEST_SET_PATH}")

## 1. Load the Trained Pipeline and Test Set

In [ ]:
pipeline = load_artifact(MODEL_PATH)
X_test, y_test = load_test_set(TEST_SET_PATH)

print(f"Loaded pipeline: {type(pipeline.named_steps['model']).__name__}")
print(f"Test set: {X_test.shape[0]} rows, default rate={y_test.mean():.4f}")

## 2. Generate Predictions

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

pd.DataFrame({
    "actual": y_test.values[:10],
    "predicted": y_pred[:10],
    "predicted_probability": y_proba[:10].round(4),
})

## 3. Core Metrics — Accuracy, Precision, Recall, F1, ROC-AUC

Recall on the Default class is the primary metric of business interest: missing a true defaulter is costlier than a false alarm.

In [ ]:
core_metrics = compute_core_metrics(y_test, y_pred, y_proba)
pd.Series(core_metrics).to_frame("value").style.format("{:.4f}")

## 4. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm_result = compute_confusion_matrix(y_test, y_pred)
print(f"True Negatives:  {cm_result['true_negatives']}")
print(f"False Positives: {cm_result['false_positives']}")
print(f"False Negatives: {cm_result['false_negatives']}")
print(f"True Positives:  {cm_result['true_positives']}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No Default", "Default"], cmap="Blues", ax=ax
)
ax.set_title("Confusion Matrix — Loan Default Model")
plt.tight_layout()
plt.show()

## 5. Classification Report

In [ ]:
class_report = compute_classification_report(y_test, y_pred)
pd.DataFrame(class_report).T.style.format("{:.4f}")

## 6. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(fpr, tpr, label=f"ROC curve (AUC = {core_metrics['roc_auc']:.4f})", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Loan Default Model")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Feature Importance

Extracted from the fitted estimator (tree-based `feature_importances_` or linear `coef_` magnitude), mapped back to human-readable feature names via the fitted `ColumnTransformer`.

In [ ]:
feature_importance = extract_feature_importance(pipeline)

if feature_importance:
    fi_df = pd.DataFrame(feature_importance)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.barplot(data=fi_df.sort_values("importance"), x="importance", y="feature", color="#2563eb", ax=ax)
    ax.set_title("Top 20 Feature Importances — Loan Default Model")
    plt.tight_layout()
    plt.show()
    display(fi_df)
else:
    print("Feature importance not available for this estimator type.")

## 8. Persist Full Evaluation Results

Runs the complete `evaluate()` workflow from `ml/src/evaluate_model.py`, writing `ml/reports/evaluation_metrics.json` and all supporting plots — this is the single source of truth for how the currently deployed `loan_pipeline.pkl` performs.

In [ ]:
results = evaluate()
print("Evaluation metrics saved to ml/reports/evaluation_metrics.json")
print(f"\nFinal Test ROC-AUC: {results['metrics']['roc_auc']:.4f}")
print(f"Final Test Recall (Default class): {results['metrics']['recall_default_class']:.4f}")

## 9. Summary

- The saved `loan_pipeline.pkl` was reloaded from disk exactly as FastAPI will at startup, and scored on the untouched held-out test set.
- Core metrics (Accuracy, Precision, Recall, F1, ROC-AUC) and the confusion matrix quantify overall performance, with particular attention to Recall on the Default class per the business priority of catching true defaulters.
- Feature importance confirms which applicant attributes drive the model's risk assessment, giving loan officers an interpretable basis for reviewing predictions.
- All results are persisted to `ml/reports/evaluation_metrics.json`, and supporting plots to `ml/reports/*.png` — these artifacts complete the Machine Learning module deliverables required before backend integration begins.